# Notebook 07: End-to-End NS-MCA Pipeline — Test Set Evaluation

**Author:** Dedeepya Korukonda (a1945558)  
**University:** University of Adelaide | COMP 6004 | May 2026  
**Purpose:** Run the complete 6-layer NS-MCA pipeline on the held-out
MedQA-USMLE test set (1,273 questions) to produce independent,
publication-standard evaluation results.

## Why the Test Set Matters

All design decisions in Notebooks 02–06 were made using the training
and development splits of MedQA-USMLE. The test split (1,273 questions)
was held out throughout and has never influenced any threshold, policy,
or recovery strategy.

Running NS-MCA on this held-out data produces results that are:
- **Independent** of all prior design decisions
- **Generalisable** beyond the training distribution
- **Publication-standard** for comparison with baselines

## Architecture Applied

$$\text{NS-MCA}(X) = \begin{cases}
\text{ACCEPT}   & S(y) = \text{true} \\
\text{RECOVER}  & S(y) = \text{false}, \text{recovery succeeds} \\
\text{ESCALATE} & S(y) = \text{false}, \text{recovery fails}
\end{cases}$$

Where:
$$S(y) \Leftrightarrow (\text{conf}(y) \geq \tau_s) \wedge
(V(y) \cap P = \emptyset)$$

## What This Notebook Produces

For each of 1,273 test questions:
- Layer 1: Neural inference + confidence
- Layer 2: Confidence calibration
- Layer 3: Entity extraction
- Layer 4: Policy auditing + S(y)
- Layer 5: Recovery attempt (if S(y)=false)
- Layer 6: Escalation (if recovery fails)

**Primary output metric:**
$$\text{Satisfiability}_{\text{test}} = \frac{N_{\text{accept}} +
N_{\text{recovered}}}{1273} \times 100\%$$

## Inputs
- `medqa_raw.json` — test split questions
- `layer2_calibration_results.json` — T* per specialty
- Flan-T5-Large model
- Policy database from Notebook 04

## Outputs
- `test_layer1_outputs.json`
- `test_layer4_results.json`
- `test_layer5_results.json`
- `test_layer6_results.json`
- `test_end_to_end_summary.json`

In [2]:
# ============================================================
# NOTEBOOK 07: END-TO-END NS-MCA PIPELINE — TEST SET
# Author: Dedeepya Korukonda (a1945558)
# Adelaide University | COMP 6004 | May 2026
# ============================================================

import json
import time
import re
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_PATH = '/content/drive/My Drive/NS-MCA-Results'
print(f"✓ Drive mounted")

# ── GPU check ─────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✓ Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print(f"  ⚠ WARNING: Switch to T4 GPU for feasible runtime.")

# ── Load MedQA test split ─────────────────────────────────────
print("\nLoading MedQA dataset...")
with open(f'{DRIVE_PATH}/medqa_raw.json',
          'r', encoding='utf-8') as f:
    medqa_raw = json.load(f)

test_questions = [q for q in medqa_raw
                  if q.get('split') == 'test']
print(f"✓ Total dataset            : {len(medqa_raw):,}")
print(f"✓ Test split extracted     : {len(test_questions):,}")

# Verify test split
if len(test_questions) == 0:
    # Fallback: use last 10% if split field missing
    n_test = int(len(medqa_raw) * 0.1)
    test_questions = medqa_raw[-n_test:]
    print(f"  (No split field found — using last {n_test} as test)")

# Assign original indices
for idx, q in enumerate(test_questions):
    if 'original_index' not in q:
        q['original_index'] = medqa_raw.index(q)

print(f"\nTest question sample:")
for q in test_questions[:3]:
    print(f"  Q: {q['question'][:70]}...")
    print(f"     Answer: {q['answer']}")

# ── Load calibration parameters from Notebook 02 ─────────────
print("\nLoading calibration parameters...")
with open(f'{DRIVE_PATH}/layer2_calibration_results.json',
          'r', encoding='utf-8') as f:
    layer2_data = json.load(f)

# Extract T* per specialty
principled = layer2_data.get(
    'temperature_principled',
    layer2_data.get('temperature_optimal', {})
)

T_STAR = {
    'general'     : principled.get('general',
                    {}).get('T_star', 1.3054),
    'pharmacology': principled.get('pharmacology',
                    {}).get('T_star', 1.5846),
    'pediatrics'  : principled.get('pediatrics',
                    {}).get('T_star', 1.5762),
    'surgery'     : principled.get('surgery',
                    {}).get('T_star', 1.5001),
}

CLINICAL_THRESHOLDS = {
    'general'     : 0.65,
    'pharmacology': 0.70,
    'pediatrics'  : 0.82,
    'surgery'     : 0.85,
}

print(f"✓ Calibration T* loaded per specialty:")
for s, t in T_STAR.items():
    tau = CLINICAL_THRESHOLDS[s]
    print(f"  {s:<15}: T*={t:.4f}, τ={tau:.2f}")

print(f"\n✓ CELL 2 COMPLETE — Test set ready ({len(test_questions)} questions)")

Mounted at /content/drive
✓ Drive mounted
✓ Device: cuda
  GPU: NVIDIA A100-SXM4-40GB

Loading MedQA dataset...
✓ Total dataset            : 12,723
✓ Test split extracted     : 1,273

Test question sample:
  Q: A junior orthopaedic surgery resident is completing a carpal tunnel re...
     Answer: Tell the attending that he cannot fail to disclose this mistake
  Q: A 67-year-old man with transitional cell carcinoma of the bladder come...
     Answer: Cross-linking of DNA
  Q: Two weeks after undergoing an emergency cardiac catherization with ste...
     Answer: Cholesterol embolization

Loading calibration parameters...
✓ Calibration T* loaded per specialty:
  general        : T*=1.3054, τ=0.65
  pharmacology   : T*=1.5846, τ=0.70
  pediatrics     : T*=1.5762, τ=0.82
  surgery        : T*=1.5001, τ=0.85

✓ CELL 2 COMPLETE — Test set ready (1273 questions)


## Cell 3: Determine Question Specialty

MedQA questions do not always have specialty labels in the raw data.
We use the `meta_info` field to assign specialty:

| meta_info value | Specialty assigned |
|----------------|-------------------|
| step1 | general |
| step2&3 | pharmacology |
| pediatrics | pediatrics |
| surgery | surgery |
| (default) | general |

This matches the specialty assignment used in Notebooks 01-06,
ensuring consistent calibration threshold application.

In [3]:
# ============================================================
# CELL 4: LOAD MODEL AND DEFINE ALL PIPELINE FUNCTIONS
# ============================================================

from transformers import T5ForConditionalGeneration, T5Tokenizer

print("=" * 60)
print("LOADING FLAN-T5-LARGE")
print("=" * 60)

MODEL_NAME = 'google/flan-t5-large'
tokenizer  = T5Tokenizer.from_pretrained(MODEL_NAME)
model      = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map='auto'
)
model.eval()
print(f"✓ Model loaded on {device}")

# ── Specialty assignment ──────────────────────────────────────
def assign_specialty(question_record):
    meta = question_record.get('meta_info', '').lower()
    q    = question_record.get('question', '').lower()
    if 'pediatric' in meta or 'pediatric' in q:
        return 'pediatrics'
    if 'surgery' in meta or 'surgical' in q[:50]:
        return 'surgery'
    if 'step2' in meta or 'step 2' in meta:
        return 'pharmacology'
    return 'general'

# ── Layer 1: Neural inference ─────────────────────────────────
def layer1_inference(question_text, model, tokenizer):
    """Generate prediction and compute confidence."""
    prompt  = f"Answer the medical question: {question_text}"
    inputs  = tokenizer(
        prompt, return_tensors='pt',
        max_length=512, truncation=True
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=64,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            return_dict_in_generate=True,
            output_scores=True
        )

    predicted = tokenizer.decode(
        outputs.sequences[0], skip_special_tokens=True
    )

    # Compute confidence: exp((1/n) * sum(log P(w_i)))
    with torch.no_grad():
        label_ids = outputs.sequences
        out       = model(input_ids=inputs, labels=label_ids)
        conf      = float(np.exp(-out.loss.item()))

    return predicted, conf

# ── Layer 2: Calibrate confidence ────────────────────────────
def layer2_calibrate(conf_raw, specialty):
    """Apply temperature scaling using T* from Notebook 02."""
    import math
    T_star   = T_STAR.get(specialty, 1.45)
    avg_lp   = math.log(max(conf_raw, 1e-10))
    conf_cal = float(np.exp(avg_lp / T_star))
    return conf_cal

# ── Layer 3: Entity extraction ────────────────────────────────
# Drug and procedure keywords from Notebook 03
DRUG_KEYWORDS = {
    'amoxicillin', 'ampicillin', 'penicillin', 'cephalexin',
    'ceftriaxone', 'ciprofloxacin', 'levofloxacin', 'azithromycin',
    'metronidazole', 'vancomycin', 'nitrofurantoin', 'trimethoprim',
    'clindamycin', 'erythromycin', 'tetracycline', 'doxycycline',
    'rifampin', 'isoniazid', 'metoprolol', 'atenolol', 'propranolol',
    'lisinopril', 'enalapril', 'captopril', 'amlodipine',
    'digoxin', 'warfarin', 'heparin', 'aspirin', 'clopidogrel',
    'atorvastatin', 'simvastatin', 'furosemide', 'spironolactone',
    'hydrochlorothiazide', 'ibuprofen', 'naproxen', 'indomethacin',
    'acetaminophen', 'morphine', 'codeine', 'oxycodone', 'fentanyl',
    'tramadol', 'naloxone', 'methadone', 'prednisone', 'prednisolone',
    'dexamethasone', 'methylprednisolone', 'metformin', 'insulin',
    'glipizide', 'risperidone', 'haloperidol', 'diazepam',
    'lorazepam', 'clonazepam', 'fluoxetine', 'sertraline',
    'paroxetine', 'venlafaxine', 'albuterol', 'theophylline',
    'cyclophosphamide', 'methotrexate', 'doxorubicin', 'cisplatin',
    'omeprazole', 'ondansetron', 'levothyroxine', 'lithium',
    'valproate', 'carbamazepine', 'phenytoin', 'colchicine',
    'oseltamivir', 'acyclovir', 'chloroquine', 'ivermectin',
    'steroid', 'antibiotic', 'nsaid', 'opioid', 'benzodiazepine',
    'statin', 'diuretic', 'antipsychotic', 'antiviral',
    'eplerenone', 'mepolizumab', 'cisplatin', 'sulfonylurea',
}

PROCEDURE_KEYWORDS = {
    'surgery', 'surgical', 'biopsy', 'catheterization',
    'endoscopy', 'colonoscopy', 'cholecystectomy', 'hysterectomy',
    'mastectomy', 'lumpectomy', 'splenectomy', 'amputation',
    'intubation', 'dialysis', 'transfusion', 'chemotherapy',
    'radiation', 'transplant', 'resection', 'excision',
    'mri', 'ct', 'ultrasound', 'echocardiography',
    'lumbar puncture', 'drainage', 'aspiration',
}

ALLERGY_MARKERS = {
    'allergy', 'allergic', 'hypersensitivity', 'anaphylaxis',
}

DOSE_PATTERN = re.compile(
    r'\b(\d+\.?\d*)\s*(mg|g|mcg|IU|units?|mEq|ml)\b',
    re.IGNORECASE
)

def layer3_extract(text):
    """Extract medical entities from prediction text."""
    text_lower = text.lower()
    entities   = {}

    # Drug extraction
    drugs = [d for d in DRUG_KEYWORDS
             if re.search(r'\b' + re.escape(d) + r'\b', text_lower)]
    if drugs:
        entities['DRUG'] = [{'name': d, 'confidence': 0.70}
                            for d in drugs]

    # Procedure extraction
    procs = [p for p in PROCEDURE_KEYWORDS
             if re.search(r'\b' + re.escape(p) + r'\b', text_lower)]
    if procs:
        entities['PROCEDURE'] = [{'name': p, 'confidence': 0.85}
                                 for p in procs]

    # Allergy flag
    if any(m in text_lower for m in ALLERGY_MARKERS):
        entities['ALLERGY_FLAG'] = True

    # Dose extraction
    doses = DOSE_PATTERN.findall(text)
    if doses:
        entities['DOSE'] = [{'value': f"{d[0]}{d[1]}",
                             'confidence': 0.95}
                            for d in doses]

    return {
        'entities'     : entities,
        'entity_count' : sum(len(v) if isinstance(v, list) else 1
                            for v in entities.values()),
        'has_drug'     : 'DRUG' in entities,
        'has_procedure': 'PROCEDURE' in entities,
    }

# ── Layer 4: Policy checking ──────────────────────────────────
OPIOID_DRUGS = {
    'morphine', 'oxycodone', 'fentanyl',
    'hydrocodone', 'codeine', 'tramadol'
}

def layer4_check(entities, conf_cal, tau_clinical):
    """
    Apply policy checking and compute S(y).
    Context-aware: only checks policies when context is present.
    """
    violations        = []
    violation_details = []
    drugs = [d['name'] for d in entities.get('DRUG', [])]

    # Opioid safety (always check — conservative)
    opioids_found = [d for d in drugs if d in OPIOID_DRUGS]
    for opioid in opioids_found:
        violations.append(f'OPIOID_SAFETY_{opioid.upper()}')
        violation_details.append({
            'category'          : 'OPIOID_SAFETY',
            'opioid_recommended': opioid,
            'severity'          : 'CRITICAL'
        })

    # Allergy contraindication (only if ALLERGY_FLAG present)
    if entities.get('ALLERGY_FLAG'):
        for drug in drugs:
            if drug in {'penicillin', 'amoxicillin', 'ampicillin',
                        'ibuprofen', 'aspirin', 'naproxen'}:
                violations.append(f'ALLERGY_{drug.upper()}')
                violation_details.append({
                    'category': 'ALLERGY_CONTRAINDICATION',
                    'drug'    : drug,
                    'severity': 'CRITICAL'
                })

    # Satisfiability gate — AND logic
    conf_passes   = conf_cal >= tau_clinical
    no_violations = len(violations) == 0
    S_y           = conf_passes and no_violations

    return {
        'S_y'              : bool(S_y),
        'conf_passes'      : bool(conf_passes),
        'violations'       : violations,
        'violation_details': violation_details,
        'num_violations'   : len(violations),
        'reason'           : (
            'ACCEPT' if S_y else
            'POLICY_VIOLATION' if violations else
            'LOW_CONFIDENCE'
        )
    }

# ── Layer 5: Recovery ─────────────────────────────────────────
def layer5_recover(question_text, predicted, violations,
                   conf_cal, model, tokenizer):
    """
    Attempt constraint-augmented recovery.
    Same logic as Notebook 05.
    """
    # Build constraint prompt
    if violations:
        prompt = (
            f"Answer the medical question carefully.\n"
            f"SAFETY CONSTRAINT: Do NOT recommend opioids, "
            f"morphine, oxycodone, fentanyl, or codeine.\n"
            f"Consider non-opioid alternatives.\n"
            f"Question: {question_text}\nAnswer:"
        )
        strategy = 'OPIOID_CONSTRAINT'
    else:
        prompt = (
            f"Answer the following medical question precisely.\n"
            f"Give a specific, concrete clinical answer.\n"
            f"Question: {question_text}\nAnswer:"
        )
        strategy = 'SPECIFICITY_BOOST'

    inputs = tokenizer(
        prompt, return_tensors='pt',
        max_length=512, truncation=True
    ).input_ids.to(device)

    with torch.no_grad():
        out_ids = model.generate(
            inputs, max_new_tokens=64,
            do_sample=False, num_beams=4
        )

    recovered_text = tokenizer.decode(
        out_ids[0], skip_special_tokens=True
    )

    # Recompute confidence
    with torch.no_grad():
        out    = model(input_ids=inputs, labels=out_ids)
        new_conf = float(np.exp(-out.loss.item()))

    # Recovery success criteria (same as Notebook 05)
    opioid_free = not any(
        o in recovered_text.lower() for o in OPIOID_DRUGS
    )
    is_meaningful = (
        len(recovered_text.strip()) > 2 and
        recovered_text.strip().lower() not in
        ['none', 'n/a', 'unknown']
    )
    medical_terms = [
        'mg', 'dose', 'treatment', 'therapy', 'diagnosis',
        'patient', 'administer', 'recommend', 'surgery',
        'monitor', 'test', 'blood', 'syndrome', 'disease'
    ]
    is_specific = (
        len(recovered_text) > len(predicted) or
        any(t in recovered_text.lower() for t in medical_terms)
    )

    if violations:
        success = opioid_free and is_meaningful
    else:
        success = is_meaningful and is_specific

    return {
        'recovered'       : bool(success),
        'recovered_text'  : recovered_text,
        'new_confidence'  : round(float(new_conf), 6),
        'strategy'        : strategy,
        'opioid_free'     : bool(opioid_free),
        'is_meaningful'   : bool(is_meaningful),
        'is_specific'     : bool(is_specific),
    }

# ── Layer 6: Escalation ───────────────────────────────────────
def layer6_escalate(violations, has_drug, has_procedure,
                    conf_cal):
    """Classify escalation severity."""
    if violations:
        return (
            'CRITICAL',
            'Opioid/policy violation not recovered',
            'Immediate physician review required'
        )
    if has_drug:
        return (
            'MEDIUM',
            f'Drug recommendation, low confidence ({conf_cal:.4f})',
            'Standard clinical review required'
        )
    if has_procedure:
        return (
            'MEDIUM',
            f'Procedure recommendation, low confidence ({conf_cal:.4f})',
            'Standard clinical review required'
        )
    return (
        'LOW',
        f'Diagnostic prediction, low confidence ({conf_cal:.4f})',
        'Routine clinical review'
    )

print(f"✓ All pipeline functions defined")
print(f"✓ CELL 4 COMPLETE — Ready to run end-to-end pipeline")

LOADING FLAN-T5-LARGE


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✓ Model loaded on cuda
✓ All pipeline functions defined
✓ CELL 4 COMPLETE — Ready to run end-to-end pipeline


## Cell 5: End-to-End Pipeline Execution

This cell runs all 6 layers sequentially on each of the 1,273
test questions. Each question passes through:
Question → L1 (inference) → L2 (calibration) → L3 (entities)
→ L4 (policy gate) → L5 (recovery if needed)
→ L6 (escalation if unrecovered)
→ Final outcome: ACCEPT / RECOVER / ESCALATE

### Expected Runtime
~15-25 minutes on A100 for 1,273 questions.

### Progress Monitoring
Progress printed every 100 questions with running totals
of accept/recover/escalate counts and current satisfiability.

In [4]:
# ============================================================
# CELL 6: END-TO-END PIPELINE ON TEST SET
# ============================================================

print("=" * 60)
print("END-TO-END NS-MCA PIPELINE: TEST SET")
print(f"Processing {len(test_questions)} questions")
print("=" * 60)

test_results = []
start_time   = time.time()

# Counters
n_accept   = 0
n_recover  = 0
n_escalate = 0
n_correct  = 0

# Per-specialty tracking
spec_stats = defaultdict(lambda: {
    'total': 0, 'accept': 0, 'recover': 0,
    'escalate': 0, 'correct': 0
})

for i, q in enumerate(test_questions):

    q_text      = q['question']
    ground_truth = q['answer']
    specialty   = assign_specialty(q)
    tau         = CLINICAL_THRESHOLDS[specialty]
    orig_idx    = q.get('original_index', i)

    # ── Layer 1: Neural inference ─────────────────────────────
    predicted, conf_raw = layer1_inference(
        q_text, model, tokenizer
    )

    # ── Layer 2: Calibration ──────────────────────────────────
    conf_cal = layer2_calibrate(conf_raw, specialty)

    # ── Layer 3: Entity extraction ────────────────────────────
    l3_result = layer3_extract(predicted)
    entities  = l3_result['entities']

    # ── Layer 4: Policy auditing ──────────────────────────────
    l4_result = layer4_check(entities, conf_cal, tau)
    S_y       = l4_result['S_y']

    # ── Check correctness against ground truth ────────────────
    is_correct = (
        ground_truth.lower().strip() in predicted.lower() or
        predicted.lower().strip() in ground_truth.lower()
    )

    # ── Layer 5: Recovery (if S(y) = false) ──────────────────
    l5_result = None
    if not S_y:
        l5_result = layer5_recover(
            q_text, predicted,
            l4_result['violations'],
            conf_cal, model, tokenizer
        )

    # ── Determine final outcome ───────────────────────────────
    if S_y:
        outcome = 'ACCEPT'
        n_accept += 1
        final_prediction = predicted
        spec_stats[specialty]['accept'] += 1

    elif l5_result and l5_result['recovered']:
        outcome = 'RECOVER'
        n_recover += 1
        final_prediction = l5_result['recovered_text']
        spec_stats[specialty]['recover'] += 1

    else:
        outcome = 'ESCALATE'
        n_escalate += 1
        final_prediction = predicted
        spec_stats[specialty]['escalate'] += 1

    if is_correct:
        n_correct += 1

    spec_stats[specialty]['total'] += 1

    # ── Layer 6: Escalation (if needed) ──────────────────────
    l6_severity = None
    l6_action   = None
    if outcome == 'ESCALATE':
        l6_severity, l6_reason, l6_action = layer6_escalate(
            l4_result['violations'],
            l3_result['has_drug'],
            l3_result['has_procedure'],
            conf_cal
        )

    # ── Store result ──────────────────────────────────────────
    result = {
        'question_id'     : orig_idx,
        'question'        : q_text[:100],
        'ground_truth'    : ground_truth,
        'specialty'       : specialty,
        'layer1': {
            'predicted'  : predicted,
            'conf_raw'   : round(float(conf_raw), 6),
        },
        'layer2': {
            'conf_cal'        : round(float(conf_cal), 6),
            'tau_clinical'    : float(tau),
            'conf_passes'     : bool(conf_cal >= tau),
        },
        'layer3': {
            'entity_count': l3_result['entity_count'],
            'has_drug'    : bool(l3_result['has_drug']),
            'has_procedure': bool(l3_result['has_procedure']),
        },
        'layer4': {
            'S_y'         : bool(S_y),
            'violations'  : l4_result['violations'],
            'num_violations': int(l4_result['num_violations']),
            'reason'      : l4_result['reason'],
        },
        'layer5': {
            'attempted'  : l5_result is not None,
            'recovered'  : bool(l5_result['recovered'])
                           if l5_result else False,
            'strategy'   : l5_result['strategy']
                           if l5_result else None,
        },
        'layer6': {
            'severity'  : l6_severity,
            'action'    : l6_action,
        },
        'outcome'        : outcome,
        'is_correct'     : bool(is_correct),
        'final_prediction': final_prediction,
    }
    test_results.append(result)

    # ── Progress ──────────────────────────────────────────────
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate    = (i + 1) / elapsed
        eta     = (len(test_questions) - i - 1) / rate
        sat     = (n_accept + n_recover) / (i + 1) * 100
        print(f"  {i+1:>5}/{len(test_questions)} | "
              f"Accept: {n_accept} | "
              f"Recover: {n_recover} | "
              f"Escalate: {n_escalate} | "
              f"Sat: {sat:.1f}% | "
              f"ETA: {eta/60:.1f}m")

total_time = time.time() - start_time
n_total    = len(test_results)

# ── Compute final metrics ─────────────────────────────────────
satisfiability = (n_accept + n_recover) / n_total * 100
accuracy       = n_correct / n_total * 100
accept_rate    = n_accept / n_total * 100
recover_rate   = n_recover / n_total * 100
escalate_rate  = n_escalate / n_total * 100

# L5 recovery rate on test set
n_l5_attempted = sum(1 for r in test_results
                     if r['layer5']['attempted'])
n_l5_recovered = sum(1 for r in test_results
                     if r['layer5']['recovered'])
test_IRR = (n_l5_recovered / n_l5_attempted
            if n_l5_attempted > 0 else 0)

# Violations on test set
n_violations = sum(1 for r in test_results
                   if r['layer4']['num_violations'] > 0)

print(f"\n{'='*60}")
print(f"TEST SET RESULTS")
print(f"{'='*60}")
print(f"\nTotal test questions : {n_total}")
print(f"Time elapsed         : {total_time/60:.1f} minutes")

print(f"\nOUTCOME DISTRIBUTION:")
print(f"  ACCEPT   : {n_accept:>5} ({accept_rate:>5.1f}%)")
print(f"  RECOVER  : {n_recover:>5} ({recover_rate:>5.1f}%)")
print(f"  ESCALATE : {n_escalate:>5} ({escalate_rate:>5.1f}%)")

print(f"\nKEY METRICS:")
print(f"  Satisfiability (test set)  : {satisfiability:.2f}%")
print(f"  Ground truth accuracy      : {accuracy:.2f}%")
print(f"  Test set IRR               : {test_IRR:.4f}")
print(f"  Policy violations detected : {n_violations}")

print(f"\nCOMPARISON WITH TRAINING SET:")
print(f"  {'Metric':<35} {'Train':>10} {'Test':>10}")
print(f"  {'-'*55}")
print(f"  {'Satisfiability':<35} {'53.6%':>10} {satisfiability:.1f}%{'':>5}")
print(f"  {'IRR':<35} {'0.5264':>10} {test_IRR:.4f}{'':>5}")

print(f"\nSPECIALTY BREAKDOWN:")
print(f"  {'Specialty':<15} {'Total':>7} {'Accept':>8} "
      f"{'Recover':>9} {'Escalate':>10} {'Sat%':>7}")
print(f"  {'-'*60}")
for spec in ['general', 'pharmacology', 'pediatrics', 'surgery']:
    s    = spec_stats[spec]
    tot  = s['total']
    if tot == 0:
        continue
    sat  = (s['accept'] + s['recover']) / tot * 100
    print(f"  {spec:<15} {tot:>7} {s['accept']:>8} "
          f"{s['recover']:>9} {s['escalate']:>10} {sat:>6.1f}%")

print(f"\n✓ CELL 6 COMPLETE")

END-TO-END NS-MCA PIPELINE: TEST SET
Processing 1273 questions
    100/1273 | Accept: 0 | Recover: 55 | Escalate: 45 | Sat: 55.0% | ETA: 24.8m
    200/1273 | Accept: 0 | Recover: 103 | Escalate: 97 | Sat: 51.5% | ETA: 23.3m
    300/1273 | Accept: 0 | Recover: 158 | Escalate: 142 | Sat: 52.7% | ETA: 21.0m
    400/1273 | Accept: 0 | Recover: 213 | Escalate: 187 | Sat: 53.2% | ETA: 19.0m
    500/1273 | Accept: 0 | Recover: 263 | Escalate: 237 | Sat: 52.6% | ETA: 16.7m
    600/1273 | Accept: 0 | Recover: 320 | Escalate: 280 | Sat: 53.3% | ETA: 14.7m
    700/1273 | Accept: 0 | Recover: 371 | Escalate: 329 | Sat: 53.0% | ETA: 12.5m
    800/1273 | Accept: 0 | Recover: 427 | Escalate: 373 | Sat: 53.4% | ETA: 10.2m
    900/1273 | Accept: 0 | Recover: 483 | Escalate: 417 | Sat: 53.7% | ETA: 8.1m
   1000/1273 | Accept: 0 | Recover: 539 | Escalate: 461 | Sat: 53.9% | ETA: 6.0m
   1100/1273 | Accept: 0 | Recover: 589 | Escalate: 511 | Sat: 53.5% | ETA: 3.8m
   1200/1273 | Accept: 0 | Recover: 642 |

## Cell 7: Save Results and Generate Plots

All test set results are saved here, along with visualisations
comparing training set and test set performance.

### Key comparison

| Metric | Training set | Test set |
|--------|-------------|---------|
| Satisfiability | 53.6% | [from Cell 6] |
| IRR | 0.5264 | [from Cell 6] |
| Accuracy | 1.26% | [from Cell 6] |

If training and test set satisfiability are within ±5pp of each
other, this confirms the architecture generalises beyond the
training distribution.

In [5]:
# ============================================================
# CELL 8: SAVE RESULTS AND GENERATE COMPARISON PLOTS
# ============================================================

print("=" * 60)
print("SAVING TEST SET RESULTS")
print("=" * 60)

# ── Build severity distribution for test set ──────────────────
test_severity_counts = defaultdict(int)
for r in test_results:
    if r['outcome'] == 'ESCALATE' and r['layer6']['severity']:
        test_severity_counts[r['layer6']['severity']] += 1

# ── Full summary ──────────────────────────────────────────────
test_summary = {
    'metadata': {
        'notebook'       : '07_EndToEnd_TestSet',
        'timestamp'      : str(time.time()),
        'n_test'         : n_total,
        'execution_time' : round(total_time / 60, 2),
        'model'          : 'google/flan-t5-large',
        'device'         : device,
    },
    'outcome_distribution': {
        'accept'  : int(n_accept),
        'recover' : int(n_recover),
        'escalate': int(n_escalate),
        'accept_pct'  : round(float(accept_rate), 2),
        'recover_pct' : round(float(recover_rate), 2),
        'escalate_pct': round(float(escalate_rate), 2),
    },
    'key_metrics': {
        'satisfiability_pct'      : round(float(satisfiability), 2),
        'ground_truth_accuracy_pct': round(float(accuracy), 2),
        'test_IRR'                : round(float(test_IRR), 4),
        'n_policy_violations'     : int(n_violations),
        'n_l5_attempted'          : int(n_l5_attempted),
        'n_l5_recovered'          : int(n_l5_recovered),
    },
    'severity_distribution': {
        sev: int(test_severity_counts[sev])
        for sev in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
    },
    'specialty_breakdown': {
        spec: {
            k: int(v) for k, v in spec_stats[spec].items()
        }
        for spec in ['general', 'pharmacology',
                     'pediatrics', 'surgery']
    },
    'comparison_with_training': {
        'train_satisfiability_pct': 53.6,
        'test_satisfiability_pct' : round(float(satisfiability), 2),
        'train_IRR'               : 0.5264,
        'test_IRR'                : round(float(test_IRR), 4),
        'generalisation_gap_pct'  : round(
            53.6 - float(satisfiability), 2
        ),
    },
    'predictions': test_results
}

# Save full results
with open(f'{DRIVE_PATH}/test_end_to_end_summary.json',
          'w', encoding='utf-8') as f:
    json.dump(test_summary, f, indent=2)
print(f"✓ Saved: test_end_to_end_summary.json")

# Save predictions only (lighter file for Notebook 08)
with open(f'{DRIVE_PATH}/test_predictions_only.json',
          'w', encoding='utf-8') as f:
    json.dump({
        'n_total'    : n_total,
        'predictions': [
            {k: v for k, v in r.items()
             if k != 'question'}
            for r in test_results
        ]
    }, f, indent=2)
print(f"✓ Saved: test_predictions_only.json")

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle(
    'NS-MCA End-to-End Results — Test Set (n=1,273)',
    fontsize=13, fontweight='bold'
)

# Plot 1: Outcome distribution
outcomes = ['ACCEPT', 'RECOVER', 'ESCALATE']
counts   = [n_accept, n_recover, n_escalate]
colors   = ['#388e3c', '#1976d2', '#f57c00']
bars     = axes[0].bar(outcomes, counts, color=colors)
axes[0].set_title('Outcome Distribution')
axes[0].set_ylabel('Number of Predictions')
for bar, cnt in zip(bars, counts):
    pct = cnt / n_total * 100
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 5,
        f'{cnt}\n({pct:.1f}%)',
        ha='center', fontsize=9
    )

# Plot 2: Train vs Test satisfiability comparison
categories = ['Layer 1\nAccuracy', 'Layer 4\nSatisfiability',
              'Layer 5\nSatisfiability (Train)',
              'Layer 5\nSatisfiability (Test)']
train_vals = [1.26, 2.10, 53.6, None]
test_vals  = [accuracy, accept_rate, None, satisfiability]

x = np.arange(len(categories))
w = 0.35
train_plot = [v if v is not None else 0 for v in train_vals]
test_plot  = [v if v is not None else 0 for v in test_vals]

axes[1].bar(x - w/2, train_plot, w,
            label='Training set', color='#1976d2', alpha=0.8)
axes[1].bar(x + w/2, test_plot,  w,
            label='Test set',     color='#388e3c', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories, fontsize=7)
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Train vs Test Comparison')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 70)

# Plot 3: Satisfiability by specialty (test set)
specs      = [s for s in ['general', 'pharmacology',
                           'pediatrics', 'surgery']
              if spec_stats[s]['total'] > 0]
sat_by_spec = [
    (spec_stats[s]['accept'] + spec_stats[s]['recover']) /
    spec_stats[s]['total'] * 100
    for s in specs
]
bar_colors = ['#1976d2', '#7b1fa2', '#f57c00', '#d32f2f']
bars3 = axes[2].bar(specs, sat_by_spec,
                    color=bar_colors[:len(specs)])
axes[2].set_title('Satisfiability by Specialty (Test)')
axes[2].set_ylabel('Satisfiability (%)')
axes[2].set_ylim(0, 80)
for bar, val in zip(bars3, sat_by_spec):
    axes[2].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{val:.1f}%', ha='center', fontsize=9
    )

plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/test_end_to_end_plots.png',
            dpi=150, bbox_inches='tight')
plt.close()
print(f"✓ Saved: test_end_to_end_plots.png")

# ── Final summary print ───────────────────────────────────────
print(f"\n{'='*60}")
print(f"✓✓✓ NOTEBOOK 07 COMPLETE ✓✓✓")
print(f"{'='*60}")
print(f"""
TEST SET RESULTS SUMMARY:
  Test questions processed   : {n_total}

  ACCEPT   (S(y)=TRUE)       : {n_accept} ({accept_rate:.1f}%)
  RECOVER  (L5 success)      : {n_recover} ({recover_rate:.1f}%)
  ESCALATE (unrecovered)     : {n_escalate} ({escalate_rate:.1f}%)

  SATISFIABILITY (test)      : {satisfiability:.2f}%
  SATISFIABILITY (train est) : 53.6%
  GENERALISATION GAP         : {53.6-satisfiability:.1f}pp

  GROUND TRUTH ACCURACY      : {accuracy:.2f}%
  TEST IRR                   : {test_IRR:.4f}
  TRAIN IRR                  : 0.5264

  Policy violations detected : {n_violations}

COMMIT MESSAGE:
  git add notebooks/07_Layer1to6_EndToEnd_TestSet.ipynb
  git commit -m "Complete: Notebook 07 - End-to-End Test Set

  Test satisfiability : {satisfiability:.2f}%
  Test IRR            : {test_IRR:.4f}
  Ground truth acc    : {accuracy:.2f}%
  Accept/Recover/Esc  : {n_accept}/{n_recover}/{n_escalate}

  Generalisation gap vs training: {53.6-satisfiability:.1f}pp"
  git push origin main

NEXT: Notebook 08 — Full Evaluation
      VPG, Satisfiability, IRR, per-specialty analysis
""")

SAVING TEST SET RESULTS
✓ Saved: test_end_to_end_summary.json
✓ Saved: test_predictions_only.json
✓ Saved: test_end_to_end_plots.png

✓✓✓ NOTEBOOK 07 COMPLETE ✓✓✓

TEST SET RESULTS SUMMARY:
  Test questions processed   : 1273
  
  ACCEPT   (S(y)=TRUE)       : 0 (0.0%)
  RECOVER  (L5 success)      : 675 (53.0%)
  ESCALATE (unrecovered)     : 598 (47.0%)
  
  SATISFIABILITY (test)      : 53.02%
  SATISFIABILITY (train est) : 53.6%
  GENERALISATION GAP         : 0.6pp
  
  GROUND TRUTH ACCURACY      : 1.02%
  TEST IRR                   : 0.5302
  TRAIN IRR                  : 0.5264
  
  Policy violations detected : 1

COMMIT MESSAGE:
  git add notebooks/07_Layer1to6_EndToEnd_TestSet.ipynb
  git commit -m "Complete: Notebook 07 - End-to-End Test Set
  
  Test satisfiability : 53.02%
  Test IRR            : 0.5302
  Ground truth acc    : 1.02%
  Accept/Recover/Esc  : 0/675/598
  
  Generalisation gap vs training: 0.6pp"
  git push origin main

NEXT: Notebook 08 — Full Evaluation
      VPG, 